# V4: clique-dual LP lower bound on the plane-port decoder (Colab)
Run the cells in order. Cell 2 copies code and data from the fork, cell 3 is a 1-round pilot (also prints the strict **Farley ceiling**), cell 4 is the long run. State is copied to Google Drive (`MyDrive/v4_rank`) every minute, so a disconnect loses at most one round: just run all cells again and it resumes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/v4_rank'
os.makedirs(OUT, exist_ok=True)
print('Drive folder:', OUT)

In [ ]:
%cd /content
!rm -rf v4 && git clone --depth 1 -b v4-plane-lb https://github.com/Matarener/integer-mult-bounds.git v4
%cd /content/v4
!pip -q install python-sat
import os, shutil
for f in ['v4_rank2_state.pkl', 'v4_rank2.progress.json', 'v4_tariff.u8', 'v4_tariff.done.json']:
    if os.path.exists(f'{OUT}/{f}'): shutil.copy(f'{OUT}/{f}', f); print('resumed from Drive:', f)
print('free RAM GB:', round(os.sysconf('SC_AVPHYS_PAGES') * os.sysconf('SC_PAGE_SIZE') / 2**30, 1), ' cores:', os.cpu_count())

In [ ]:
# helper: run the LP script, stream its log here, copy state to Drive every 60 s
import subprocess, threading, time, shutil, os, sys
def sync():
    for f in ['v4_rank2_state.pkl', 'v4_rank2.progress.json', 'v4_rank2_result.json', 'v4_rank2.log']:
        if os.path.exists(f):
            try: shutil.copy(f, f'{OUT}/{f}.tmp'); os.replace(f'{OUT}/{f}.tmp', f'{OUT}/{f}')
            except Exception as e: print('sync error', e)
def run(max_iters, hours):
    env = dict(os.environ, V4_MAX_ITERS=str(max_iters), V4_TIME_LIMIT_SEC=str(int(hours * 3600)))
    stop = threading.Event()
    def loop():
        while not stop.wait(60): sync()
    threading.Thread(target=loop, daemon=True).start()
    with open('v4_rank2.log', 'a') as log:
        p = subprocess.Popen([sys.executable, '-u', 'v4_rank_lp2.py'], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            log.write(line); log.flush()
            if 'tariff rows' not in line: print(line, end='')
        p.wait()
    stop.set(); sync()
    print('exit code', p.returncode)
    # keep the tariff matrix on Drive too (1.5 GB, saves ~3 min on the next start)
    if not os.path.exists(f'{OUT}/v4_tariff.u8') and os.path.exists('v4_tariff.u8'):
        shutil.copy('v4_tariff.u8', f'{OUT}/v4_tariff.u8'); shutil.copy('v4_tariff.done.json', f'{OUT}/v4_tariff.done.json')


## Pilot: one round (check speed, Farley ceiling, strict certificate)
Look for the line `FARLEY CEILING: final LP <= ...` and at the end `DONE strict LB ...`.

In [ ]:
run(max_iters=1, hours=2)

## Long run
Resumes from the pilot. Keep this tab open (Colab stops idle sessions). Each round prints one `{'iter': ...}` line; every 10th round prints a new strict Farley ceiling.

In [ ]:
run(max_iters=100000, hours=10)